# This notebook walks us through 2 algorithms on same fraud detection use case data set
# First : 2 qubit  VQC - Quantum classification 
# Second : Xgboost classification

# Nature of the dataset :fraud_detection_dataset.csv 2,000 rows; exactly 40 fraud cases (2%)

transaction_amount
- The monetary value of a transaction.
- Larger or unusual amounts can be a fraud signal.
- It is included to make the dataset look like a normal payment-fraud dataset, but the current fair VQC/XGBoost scripts do not use it.

device_trust_score
- A value from 0 to 1 representing how trustworthy or familiar the customer’s device appears.
- Near 1 means more trusted/known.
- Near 0 means less trusted or more unusual.
- This is one of the two features used by both the VQC and XGBoost benchmarks.

merchant_risk_score
- A value from 0 to 1 representing the estimated risk level of the merchant.
- Near 1 means a higher-risk merchant category or behaviour.
- Near 0 means lower merchant risk.
- This is the second feature used by both benchmark models.

location_distance_km
- Approximate distance, in kilometres, from the customer’s usual transaction location.
- A large distance may be suspicious.
- It is present as a realistic classical fraud feature, but not used in the two-feature benchmark.

transactions_last_hour
- The number of transactions associated with the customer in the previous hour.
- A high value can indicate rapid repeated attempts, which may be fraud.
- It is also included for realism but not used in the current two-feature comparison.

fraud
- The target label.
- 1 means fraudulent transaction.
- 0 means legitimate transaction.
- There are 40 fraud transactions out of 2,000 rows: exactly 2% fraud.

please note  we will perform VQC and XGboost benchmarks on this data set , only on 2 features 

device_trust_score, merchant_risk_score 

VQC uses  1 qubit per feature , XGboost  gets these 2 inputs 


# Finally : we will discuss  why VQC over XGboost ?

## 0. Setup

Install the packages once in the same notebook environment:

```python
pip install "qiskit>=2.0" qiskit-machine-learning qiskit-algorithms pennylane scikit-learn scipy matplotlib
```


In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import average_precision_score, classification_report, confusion_matrix
from sklearn.utils import resample


In [2]:

SEED = 31
rng = np.random.default_rng(SEED)

In [3]:
DATA_PATH = "fraud_detection_dataset.csv"

## 1. Load and understand the classical data

The two selected features are:

- `device_trust_score`: near 1 means a more familiar/trusted device;
- `merchant_risk_score`: near 1 means a riskier merchant.

We use two features because this first VQC uses two qubits: one feature is encoded on each qubit. The other CSV fields are realistic context, but are intentionally excluded here so that the circuit stays small and understandable.

In [4]:
df = pd.read_csv(DATA_PATH)
FEATURES = ["device_trust_score", "merchant_risk_score"]
TARGET = "fraud"

print("Rows:", len(df))
print("Fraud rate:", f"{df[TARGET].mean():.1%}")
display(df.head())

Rows: 2000
Fraud rate: 2.0%


,transaction_amount,device_trust_score,merchant_risk_score,location_distance_km,transactions_last_hour,fraud
0,29.481383,0.841232,0.161340,9.935131,1,0
1,49.955465,0.454358,0.109672,10.918679,0,0
2,65.739674,0.740248,0.318506,15.368951,0,0
3,18.583464,0.667374,0.044560,26.339296,2,1
4,74.748669,0.478253,0.526545,23.507548,0,0


## 2. Classical preprocessing before quantum encoding

The VQC cannot accept a raw feature such as `0.74` as a transaction field. It needs a gate angle. `MinMaxScaler` maps both selected classical features to the interval ([-pi,pi]).

We fit the scaler on training data only. That prevents information from the test set leaking into training.

The dataset has only 2% fraud. To stop the VQC from learning the trivial "always legitimate" rule, we oversample fraud **only in training**. The test set remains naturally imbalanced.

In [5]:
X_raw = df[FEATURES].to_numpy()
y = df[TARGET].to_numpy()

X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X_raw, y, test_size=0.30, random_state=SEED, stratify=y
)

scaler = MinMaxScaler(feature_range=(-np.pi, np.pi)).fit(X_train_raw)
X_train = scaler.transform(X_train_raw)
X_test = scaler.transform(X_test_raw)

fraud_idx = np.flatnonzero(y_train == 1)
legit_idx = np.flatnonzero(y_train == 0)

#the below line  duplicates fraud-row indices until the number of fraud training examples matches the number of legitimate training examples.
fraud_upsampled = resample(
    fraud_idx, replace=True, n_samples=len(legit_idx), random_state=SEED
)

#below line creates a balanced fitting set: roughly 50% fraud and 50% legitimate.
fit_idx = np.concatenate([legit_idx, fraud_upsampled])
rng.shuffle(fit_idx)

# A small subset keeps statevector optimisation practical on a laptop.
fit_idx = rng.choice(fit_idx, size=min(240, len(fit_idx)), replace=False)
X_fit, y_fit = X_train[fit_idx], y_train[fit_idx]

print("Fraud rate in original training data:", f"{y_train.mean():.1%}")
print("Fraud rate in VQC fitting subset:", f"{y_fit.mean():.1%}")
print("Fraud rate in untouched test data:", f"{y_test.mean():.1%}")

Fraud rate in original training data: 2.0%
Fraud rate in VQC fitting subset: 47.5%
Fraud rate in untouched test data: 2.0%


# Important: this happens only for X_fit, y_fit—the VQC training subset. The test set stays at its real 2% fraud rate, so PR-AUC, precision, and recall are evaluated realistically.

## 3. A shared evaluation function

For a 2%-fraud dataset, accuracy is misleading: always predicting "legitimate" is already around 98% accurate. We focus on:

- **PR-AUC**: how well the model ranks fraud above legitimate transactions;
- **recall for class 1**: how much actual fraud it catches;
- **precision for class 1**: how trustworthy flagged cases are;
- the confusion matrix: counts of true/false positives and negatives.

In [6]:
def evaluate(name, y_true, probability_fraud, threshold=0.50):
    probability_fraud = np.asarray(probability_fraud, dtype=float)
    prediction = (probability_fraud >= threshold).astype(int)
    print(f"\n{name}")
    print("PR-AUC:", round(average_precision_score(y_true, probability_fraud), 4))
    print("Confusion matrix [ [TN, FP], [FN, TP] ]:\n", confusion_matrix(y_true, prediction))
    print(classification_report(y_true, prediction, digits=3, zero_division=0))
    return prediction

# Part A - Plain Qiskit VQC

## 4. Feature map, ansatz, and observable

A VQC has three conceptual pieces:

1. **Feature map:** Encodes each transaction’s two scaled feature values into a two-qubit quantum state. `zz_feature_map` also includes interaction terms between the two features.

2. **Ansatz:** The trainable circuit. `real_amplitudes` uses trainable $R_y(\theta)$ rotations and entangling CNOT gates.

3. **Observable/readout:** We measure the average Pauli-Z expectation value. Since

$$
\langle Z \rangle \in [-1, 1],
$$

we convert it into a fraud probability:

$$
P(\text{fraud}) = P(|1\rangle)
= \frac{1 - \langle Z \rangle}{2}.
$$

Pauli-Z is chosen because its eigenstates are the ordinary computational-basis states:

$$
Z|0\rangle = +|0\rangle,
\qquad
Z|1\rangle = -|1\rangle.
$$

Therefore, $|0\rangle$ has eigenvalue $+1$, while $|1\rangle$ has eigenvalue $-1$.

In [7]:
from qiskit.circuit.library import zz_feature_map, real_amplitudes
from qiskit.quantum_info import SparsePauliOp, Statevector
from scipy.optimize import minimize

NUM_QUBITS = 2
feature_map = zz_feature_map(feature_dimension=NUM_QUBITS, reps=2, entanglement="linear")
ansatz = real_amplitudes(num_qubits=NUM_QUBITS, reps=2, entanglement="linear")
circuit = feature_map.compose(ansatz)

# Average Z readout across both qubits.
observable = (SparsePauliOp("IZ") + SparsePauliOp("ZI")) / 2

print("Feature-map parameters:", list(feature_map.parameters))
print("Trainable ansatz parameters:", list(ansatz.parameters))
circuit.decompose().draw("mpl", style="iqx", scale=1.15)
plt.show()

Feature-map parameters: [ParameterVectorElement(x[0]), ParameterVectorElement(x[1])]
Trainable ansatz parameters: [ParameterVectorElement(θ[0]), ParameterVectorElement(θ[1]), ParameterVectorElement(θ[2]), ParameterVectorElement(θ[3]), ParameterVectorElement(θ[4]), ParameterVectorElement(θ[5])]


c:\Users\Asus\Documents\agentic_ai_use_cases\venv\lib\site-packages\qiskit\visualization\circuit\matplotlib.py:272: UserWarning: Style JSON file 'iqx.json' not found in any of these locations: c:\Users\Asus\Documents\agentic_ai_use_cases\venv\lib\site-packages\qiskit\visualization\circuit\styles\iqx.json, iqx.json. Will use default style.
  self._style, def_font_ratio = load_style(


In the above code, `zz_feature_map` takes our scaled input values and inserts them into parameterised rotation gates—mainly $R_z$-based rotations and ZZ interaction terms.

The roles of the other lines are:

- `real_amplitudes(...)`
  - This is the trainable ansatz.
  - Its parameters are model weights, not input data.
  - It applies trainable $R_y(\theta)$ gates and CNOT-based entanglement.

- `feature_map.compose(ansatz)`
  - Puts the two pieces in sequence:

$$
\text{angle-encoded data}
\rightarrow
\text{trainable VQC}
\rightarrow
\text{Pauli-Z measurement}
$$

- `SparsePauliOp("IZ")` and `SparsePauliOp("ZI")`
  - These define the Pauli-Z readout observable.
  - They do not encode data; they measure the final circuit state.

## 5. Plain Qiskit prediction and loss

For each transaction input $x$, we bind the two scaled data values to the feature-map parameters and the trainable weights $\theta$ to the ansatz parameters.

`Statevector.from_instruction` simulates the final quantum state produced by the complete circuit. We then calculate the Pauli-Z expectation value,

$$
\langle Z \rangle,
$$

and convert it into the probability that the transaction belongs to the fraud class:

$$
P(\text{fraud}) = \frac{1 - \langle Z \rangle}{2}.
$$

We use binary cross entropy as the loss function:

$$
L = -\frac{1}{N}\sum_{i=1}^{N}
\left[
y_i \log(p_i)
+
(1-y_i)\log(1-p_i)
\right].
$$

Here, $y_i$ is the actual fraud label and $p_i$ is the VQC-predicted fraud probability.

The classical optimiser changes only the ansatz weights $\theta$. The feature values are not trained; they remain the transaction inputs.

In [8]:
def qiskit_probability_fraud(x, weights):
    parameter_values = dict(zip(feature_map.parameters, x))
    parameter_values.update(dict(zip(ansatz.parameters, weights)))
    state = Statevector.from_instruction(circuit.assign_parameters(parameter_values))
    expected_z = float(np.real(state.expectation_value(observable)))
    return np.clip((1.0 - expected_z) / 2.0, 1e-6, 1 - 1e-6)

def qiskit_cross_entropy(weights, X_values, labels):
    probabilities = np.array([qiskit_probability_fraud(x, weights) for x in X_values])
    return -np.mean(labels*np.log(probabilities) + (1-labels)*np.log(1-probabilities))

initial_weights = rng.uniform(-np.pi, np.pi, ansatz.num_parameters)
result = minimize(
    qiskit_cross_entropy, initial_weights, args=(X_fit, y_fit),
    method="COBYLA", options={"maxiter": 300}
)

p_plain_qiskit = np.array([qiskit_probability_fraud(x, result.x) for x in X_test])
evaluate("Plain Qiskit VQC", y_test, p_plain_qiskit)


Plain Qiskit VQC
PR-AUC: 0.0416
Confusion matrix [ [TN, FP], [FN, TP] ]:
 [[236 352]
 [  1  11]]
              precision    recall  f1-score   support

           0      0.996     0.401     0.572       588
           1      0.030     0.917     0.059        12

    accuracy                          0.412       600
   macro avg      0.513     0.659     0.315       600
weighted avg      0.976     0.412     0.562       600



array([0, 1, 0, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 1, 1, 0, 1, 1, 1, 1, 1,
       1, 1, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 1, 1, 1, 1, 0,
       1, 0, 0, 1, 0, 0, 1, 0, 1, 0, 0, 1, 1, 1, 0, 1, 1, 1, 1, 0, 0, 1,
       1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 1, 1, 0, 1, 0, 0, 1, 0, 1, 1, 0, 1,
       1, 0, 1, 0, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 0, 1, 0, 1,
       1, 1, 0, 1, 1, 0, 1, 1, 1, 1, 1, 0, 0, 1, 1, 1, 1, 1, 0, 1, 1, 1,
       0, 0, 1, 0, 1, 1, 1, 0, 1, 1, 0, 0, 1, 1, 1, 1, 1, 0, 0, 1, 1, 0,
       1, 1, 1, 0, 1, 1, 0, 1, 1, 1, 0, 0, 1, 1, 0, 1, 0, 0, 1, 0, 1, 0,
       1, 1, 1, 0, 1, 1, 1, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 0,
       0, 1, 0, 0, 1, 1, 1, 0, 1, 1, 1, 0, 1, 0, 1, 1, 1, 0, 0, 1, 1, 1,
       0, 1, 0, 1, 1, 0, 1, 1, 0, 0, 1, 0, 1, 1, 0, 0, 1, 0, 1, 0, 1, 1,
       1, 1, 0, 1, 1, 1, 0, 0, 0, 1, 1, 1, 1, 0, 0, 1, 0, 0, 1, 1, 0, 0,
       0, 0, 0, 0, 0, 1, 1, 0, 1, 1, 0, 1, 1, 1, 0, 1, 1, 1, 1, 0, 1, 0,
       0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1,

# Part B - Qiskit Machine Learning

## 6. The same VQC with the Qiskit Machine Learning wrapper

The previous section made every piece visible. Qiskit Machine Learning packages the same idea into its `VQC` classifier:

- you provide a feature map and ansatz;
- the library builds the quantum neural-network classifier;
- it applies the specified loss and classical optimiser.

The data preparation and test set remain unchanged, so this is comparable with the plain-Qiskit section.

In [ ]:
#! pip install qiskit-algorithms

     ---------------------------------------- 0.0/327.8 kB ? eta -:--:--
     ------------------ ------------------- 163.8/327.8 kB 3.3 MB/s eta 0:00:01
     -------------------------------------- 327.8/327.8 kB 4.1 MB/s eta 0:00:00



[notice] A new release of pip is available: 23.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [9]:
from qiskit_algorithms.optimizers import COBYLA
from qiskit_machine_learning.algorithms.classifiers import VQC

In [10]:

## Below line creates qiskit machine learning vqc clasifier  object 

qml_vqc = VQC(
    feature_map=feature_map,
    ansatz=ansatz,
    loss="cross_entropy",
    optimizer=COBYLA(maxiter=300),
)
qml_vqc.fit(X_fit, y_fit)

# Column 1 is the model probability assigned to fraud label 1.
p_qiskit_ml = qml_vqc.predict_proba(X_test)[:, 1]
evaluate("Qiskit Machine Learning VQC", y_test, p_qiskit_ml)


Qiskit Machine Learning VQC
PR-AUC: 0.0267
Confusion matrix [ [TN, FP], [FN, TP] ]:
 [[258 330]
 [  2  10]]
              precision    recall  f1-score   support

           0      0.992     0.439     0.608       588
           1      0.029     0.833     0.057        12

    accuracy                          0.447       600
   macro avg      0.511     0.636     0.333       600
weighted avg      0.973     0.447     0.597       600



array([0, 0, 1, 1, 1, 0, 0, 1, 1, 1, 0, 0, 0, 0, 1, 0, 0, 1, 0, 1, 0, 1,
       1, 1, 0, 0, 0, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 1, 1, 1, 1,
       1, 0, 0, 1, 0, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 1,
       0, 1, 0, 1, 0, 0, 1, 0, 1, 1, 1, 1, 1, 1, 0, 1, 1, 0, 1, 1, 0, 1,
       1, 1, 1, 0, 1, 1, 0, 1, 1, 1, 1, 0, 0, 1, 1, 0, 1, 0, 1, 1, 0, 1,
       1, 1, 0, 0, 1, 0, 0, 0, 0, 1, 1, 0, 0, 1, 1, 0, 1, 1, 0, 1, 0, 1,
       1, 0, 0, 1, 1, 1, 1, 0, 0, 1, 0, 1, 0, 1, 1, 1, 0, 0, 1, 1, 1, 1,
       1, 1, 1, 1, 1, 1, 0, 1, 1, 0, 0, 0, 1, 0, 0, 1, 1, 0, 1, 0, 0, 0,
       1, 1, 1, 0, 0, 1, 1, 1, 0, 1, 0, 1, 1, 0, 0, 0, 0, 1, 1, 1, 0, 1,
       0, 1, 0, 1, 0, 0, 1, 0, 1, 0, 1, 0, 0, 0, 0, 1, 1, 0, 0, 1, 0, 1,
       0, 1, 0, 1, 1, 0, 0, 1, 1, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 1,
       1, 1, 0, 1, 1, 1, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1,
       0, 0, 0, 1, 0, 1, 1, 0, 1, 0, 0, 1, 0, 1, 1, 0, 1, 0, 1, 0, 1, 1,
       0, 1, 1, 0, 0, 0, 0, 1, 1, 0, 0, 1, 1, 1, 0,

# Part C - PennyLane VQC

## 7. The same learning idea in PennyLane

PennyLane expresses a quantum circuit as a `QNode`. The procedure is unchanged:

- `AngleEmbedding` loads the two scaled classical values as (R_y)-style rotation angles;
- `StronglyEntanglingLayers` is the trainable ansatz;
- the circuit returns a Pauli-Z expectation;
- Adam adjusts the trainable weights to reduce binary cross entropy.

PennyLane makes differentiation through quantum circuits particularly explicit.

In [11]:
#!pip install pennylane

In [12]:
import pennylane as qml
import numpy as np
from scipy.optimize import minimize

device = qml.device("default.qubit", wires=NUM_QUBITS)
NUM_LAYERS = 2

@qml.qnode(device, interface=None)
def pennylane_circuit(x, weights):
    qml.AngleEmbedding(x, wires=range(NUM_QUBITS), rotation="Y")
    qml.StronglyEntanglingLayers(weights, wires=range(NUM_QUBITS))
    return qml.expval(qml.PauliZ(0))


def pennylane_probability_fraud(x, flat_weights):
    weights_3d = flat_weights.reshape(NUM_LAYERS, NUM_QUBITS, 3)
    expected_z = float(pennylane_circuit(x, weights_3d))
    return np.clip((1 - expected_z) / 2, 1e-7, 1 - 1e-7)


def pennylane_loss(flat_weights, X_values, labels):
    probabilities = np.array([
        pennylane_probability_fraud(x, flat_weights)
        for x in X_values
    ])

    return -np.mean(
        labels * np.log(probabilities)
        + (1 - labels) * np.log(1 - probabilities)
    )

In [13]:
initial_weights = rng.uniform(-np.pi, np.pi, NUM_LAYERS * NUM_QUBITS * 3)

result_pennylane = minimize(
    pennylane_loss,
    initial_weights,
    args=(X_fit, y_fit),
    method="COBYLA",
    options={"maxiter": 300}
)

p_pennylane = np.array([
    pennylane_probability_fraud(x, result_pennylane.x)
    for x in X_test
])

evaluate("PennyLane VQC", y_test, p_pennylane)

final_weights = result_pennylane.x.reshape(
    NUM_LAYERS, NUM_QUBITS, 3
)

print(qml.draw(pennylane_circuit)(X_fit[0], final_weights))


PennyLane VQC
PR-AUC: 0.3979
Confusion matrix [ [TN, FP], [FN, TP] ]:
 [[395 193]
 [  0  12]]
              precision    recall  f1-score   support

           0      1.000     0.672     0.804       588
           1      0.059     1.000     0.111        12

    accuracy                          0.678       600
   macro avg      0.529     0.836     0.457       600
weighted avg      0.981     0.678     0.790       600

0: ─╭AngleEmbedding(M0)─╭StronglyEntanglingLayers(M1)─┤  <Z>
1: ─╰AngleEmbedding(M0)─╰StronglyEntanglingLayers(M1)─┤     

M0 = 
[ 0.40634526 -0.59716613]
M1 = 
[[[-0.92649078 -0.83120527  3.5287145 ]
  [ 2.9397387   2.85984349  0.47444777]]

 [[-0.52649513  2.63950966 -2.95828955]
  [-0.70850167  2.43832097 -0.18290215]]]


## 8. Compare the three implementations

All three sections follow the same VQC logic, but expose different levels of abstraction:

| Approach | Best for | What you control directly |
|---|---|---|
| Plain Qiskit | Learning how a VQC works | Parameter binding, statevector, observable and custom loss |
| Qiskit Machine Learning | Faster Qiskit experimentation | Feature map, ansatz, optimiser and classifier settings |
| PennyLane | Differentiable quantum programming | QNode, measurement, gradient-based optimiser and circuit layers |

Small differences in scores are expected because the ansatz, optimiser, starting weights and training procedures are not identical.

In [14]:
summary = pd.DataFrame({
    "model": ["Plain Qiskit VQC", "Qiskit ML VQC", "PennyLane VQC"],
    "PR_AUC": [
        average_precision_score(y_test, p_plain_qiskit),
        average_precision_score(y_test, p_qiskit_ml),
        average_precision_score(y_test, p_pennylane),
    ],
})
display(summary.sort_values("PR_AUC", ascending=False))


,model,PR_AUC
2,PennyLane VQC,0.397866
0,Plain Qiskit VQC,0.041646
1,Qiskit ML VQC,0.026653


But `0.3564` is not very low for a dataset where only 2% of transactions are fraud.

The baseline PR-AUC of a random classifier is approximately the fraud rate:

$$
\text{Random PR-AUC} \approx 0.02
$$

Therefore:

$$
0.3564 \gg 0.02
$$

This means that the PennyLane model ranks fraud cases much better than a random classifier on this particular test split. It is below the theoretical maximum of `1.0`, but it is still a meaningful result.

However, we should not yet conclude that PennyLane is inherently better than Qiskit. The three VQC implementations are not identical:

- PennyLane uses a different ansatz: `StronglyEntanglingLayers`.
- PennyLane reads Pauli-Z from qubit 0 only.
- PennyLane uses the SciPy COBYLA optimiser.
- The Qiskit implementations use `ZZFeatureMap` and `RealAmplitudes`.

Therefore, the current result means:

> This particular PennyLane circuit configuration achieved the highest PR-AUC on this synthetic dataset and this test split.

It does not prove that PennyLane itself is superior to Qiskit.

#  Apply Classical classification algorithm XGboost  on the same  fraud dataset 

In [15]:
#!pip install xgboost scikit-learn pandas

In [16]:
# XGBoost on the same normal-looking dataset and same two inputs as the VQC.
# Install: pip install xgboost scikit-learn pandas
import pandas as pd
from xgboost import XGBClassifier
from sklearn.metrics import average_precision_score, roc_auc_score, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split

In [17]:
df = pd.read_csv("fraud_detection_dataset.csv")
FEATURES = ["device_trust_score", "merchant_risk_score"]
X, y = df[FEATURES], df.fraud
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=.30, random_state=31, stratify=y)
weight = (y_train == 0).sum() / (y_train == 1).sum()

In [18]:
xgb = XGBClassifier(n_estimators=100, max_depth=3, learning_rate=.07, subsample=.85,
                    scale_pos_weight=weight, eval_metric="logloss", random_state=31, n_jobs=-1)
xgb.fit(X_train, y_train)
prob = xgb.predict_proba(X_test)[:, 1]; pred = (prob >= .50).astype(int)
print("XGBoost PR-AUC :", round(average_precision_score(y_test, prob), 4))
print("XGBoost ROC-AUC:", round(roc_auc_score(y_test, prob), 4))
print("Confusion matrix [ [TN, FP], [FN, TP] ]:\n", confusion_matrix(y_test, pred))
print(classification_report(y_test, pred, digits=3, zero_division=0))

XGBoost PR-AUC : 0.4103
XGBoost ROC-AUC: 0.9742
Confusion matrix [ [TN, FP], [FN, TP] ]:
 [[561  27]
 [  1  11]]
              precision    recall  f1-score   support

           0      0.998     0.954     0.976       588
           1      0.289     0.917     0.440        12

    accuracy                          0.953       600
   macro avg      0.644     0.935     0.708       600
weighted avg      0.984     0.953     0.965       600



In [19]:
p_xgboost = prob

# now get the classical + vqc(quantum)  display metrics

In [20]:
summary = pd.DataFrame({
    "model": [
        "Plain Qiskit VQC",
        "Qiskit ML VQC",
        "PennyLane VQC",
        "XGBoost",
    ],
    "PR_AUC": [
        average_precision_score(y_test, p_plain_qiskit),
        average_precision_score(y_test, p_qiskit_ml),
        average_precision_score(y_test, p_pennylane),
        average_precision_score(y_test, p_xgboost),
    ],
})

display(summary.sort_values("PR_AUC", ascending=False))

,model,PR_AUC
3,XGBoost,0.410253
2,PennyLane VQC,0.397866
0,Plain Qiskit VQC,0.041646
1,Qiskit ML VQC,0.026653


## Final analysis

On this particular test split, XGBoost performs better than the PennyLane VQC:

$$
\text{XGBoost PR-AUC} = 0.4103
$$

$$
\text{PennyLane VQC PR-AUC} = 0.3564
$$

This is a reasonable outcome. XGBoost is a strong classical model for structured tabular data and can capture nonlinear feature relationships effectively.

However, we should not overinterpret this difference. Our test set contains only **12 fraud transactions**. With such a small number of positive examples, the ranking of even one or two fraud cases can noticeably change the PR-AUC score.

### Should we use amplitude encoding?

For this experiment, **angle encoding is more appropriate than amplitude encoding**. We use only two selected input features, so each feature can be mapped directly to one qubit.

Amplitude encoding is not recommended here because:

- it normalizes the two features together;
- this normalization may reduce useful scale information;
- it requires more complex quantum state preparation;
- it provides no meaningful qubit-saving advantage for only two input features.

### Fair VQC improvements to try

The following experiments may improve the VQC, while keeping the comparison with XGBoost fair.

1. **Increase circuit depth gradually**

```python
NUM_LAYERS = 3

Use matching initial weights:

```python
initial_weights = rng.normal(
    loc=0,
    scale=0.05,
    size=NUM_LAYERS * NUM_QUBITS * 3
)

More layers make the VQC more expressive, but too many layers can overfit a small dataset.

2. Increase the COBYLA optimisation budget
options={"maxiter": 250}


The current setting of 100 iterations may not be enough for the optimiser to find good circuit parameters.

3. Repeat training with multiple random seeds
VQCs may converge to different local optima depending on their initial weights. We should run several seeds and report the mean and standard deviation of PR-AUC, rather than reporting only the best run.

4. Use a validation set for tuning
We should tune the number of layers, optimiser iterations, and classification threshold using a validation set. The final test set should remain untouched until final evaluation.



5. **Try data re-uploading**

Data re-uploading means encoding the same input features more than once within the quantum circuit:

$$
\text{encode data}
\rightarrow
\text{trainable layer}
\rightarrow
\text{encode data again}
\rightarrow
\text{trainable layer}
$$

This gives the VQC repeated access to the transaction features while it applies trainable transformations. It can help a small VQC learn more complex nonlinear decision boundaries.



6. Tune the fraud-alert threshold
threshold = 0.70pred = (p_pennylane >= threshold).astype(int)


Threshold tuning does not change PR-AUC, but it can improve the practical balance between fraud recall and false-positive alerts.

7. **Use repeated stratified train-test splits**

Because the dataset contains only 40 fraud examples, results from one train-test split may vary considerably. We should evaluate both models across several stratified splits and report:

$$
\text{mean PR-AUC} \pm \text{standard deviation}
$$

Stratification preserves the approximately 2% fraud rate in every training and test split.

For a fair comparison, XGBoost and the VQC must use the same input features, the same splits, and the same evaluation metrics. If XGBoost continues to outperform the VQC after fair tuning, that is a valid experimental conclusion.